[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/<USERNAME>/<REPO>/blob/main/notebooks/practicos/jax/UD4_Practico_Optimizadores_JAX_Equinox.ipynb)
markdown
markdown
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/<USERNAME>/<REPO>/blob/main/notebooks/practicos/jax/UD4_Practico_Optimizadores_JAX_Equinox.ipynb)
\n
Replace `<USERNAME>/<REPO>` with your GitHub user and repository path to make the badge open this notebook directly in Colab.
markdown
markdown
**One-line install (copy for students):**

```bash
mamba create -n ud4-optimizers -y python=3.10 && conda activate ud4-optimizers && mamba install -y -c conda-forge numpy matplotlib scikit-learn && mamba install -n ud4-optimizers -y pytorch torchvision cpuonly -c pytorch

# Alternativa pip:
pip install -r requirements.txt
```
**Instalación local (recomendado con conda/mamba)**:

- Crear entorno con `mamba`/`conda` y activar:
```bash
mamba create -n ud4-optimizers -y python=3.10
conda activate ud4-optimizers
```
- Dependencias principales (CPU):
```bash
mamba install -y -c conda-forge numpy matplotlib scikit-learn
# Equinox/Optax/JAX (CPU):
pip install equinox optax jax jaxlib
```
- Para GPU: instala la rueda `jaxlib` con soporte CUDA según https://github.com/google/jax#pip-installation y la variante de PyTorch/torchvision para CUDA si la usas.
- Alternativa `pip`: `pip install -r requirements.txt` (ajusta `requirements.txt` según el entorno).

# Optimizadores en JAX con Equinox y Optax

Este notebook muestra un MLP simple implementado con Equinox (`equinox`) y optimizado con `optax`.
Incluye instrucciones para ejecutar en Colab (instalación de jax/`jaxlib` según CPU/GPU).

In [ ]:
# Colab install helper (run in Colab if needed)
import sys
def install_colab():
    print('If running in Colab, run these commands before executing:')
    print('!pip install -q equinox optax jax jaxlib')

install_colab()

In [ ]:
import jax
import jax.numpy as jnp
import equinox as eqx
import optax
import numpy as np

class MLP(eqx.Module):
    layers: list
    def __init__(self, in_dim, hidden, out_dim, key):
        keys = jax.random.split(key, len(hidden)+1)
        self.layers = []
        dims = [in_dim] + hidden + [out_dim]
        for i in range(len(dims)-1):
            k = keys[i]
            w = jax.random.normal(k, (dims[i], dims[i+1])) * jnp.sqrt(2.0/dims[i])
            b = jnp.zeros((dims[i+1],))
            self.layers.append((w,b))

def __call__(self, x):
    for w,b in self.layers[:-1]:
        x = jnp.dot(x, w) + b
        x = jax.nn.relu(x)
    w,b = self.layers[-1]
    x = jnp.dot(x, w) + b
    return x

In [ ]:
# Toy dataset: regression y = sum(x) + noise
key = jax.random.PRNGKey(0)
N = 256
D = 16
X = jax.random.normal(key, (N,D))
y = jnp.sum(X, axis=1, keepdims=True) + 0.1 * jax.random.normal(key, (N,1))

model = MLP(D, [64,32], 1, key)
opt = optax.adam(1e-3)
opt_state = opt.init(eqx.filter(model, eqx.is_array))

@jax.jit
def loss_and_grads(model, xb, yb):
    def loss_fn(m):
        preds = m(xb)
        return jnp.mean((preds - yb)**2)
    loss, grads = jax.value_and_grad(loss_fn)(model)
    return loss, grads

@jax.jit
def update(model, opt_state, xb, yb):
    loss, grads = loss_and_grads(model, xb, yb)
    grads_filt = eqx.filter(grads, eqx.is_array)
    updates, new_opt_state = opt.update(grads_filt, opt_state, eqx.filter(model, eqx.is_array))
    new_model = eqx.apply_updates(model, updates)
    return new_model, new_opt_state, loss

# Training loop
batch_size = 32
for epoch in range(5):
    perm = jax.random.permutation(key, N)
    for i in range(0, N, batch_size):
        idx = perm[i:i+batch_size]
        xb = X[idx]
        yb = y[idx]
        model, opt_state, loss = update(model, opt_state, xb, yb)
    print('Epoch', epoch, 'loss', float(loss))